# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tenuka-R/FlyRank-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule is to score a content page as a refresh opportunity if it shows three signs - staleness, good position and low click through rate. The idea behind these combined signs is that a page that ranks well but only gets a little number of views is more likely to have a title issue instead of a genuine relevance issue.
The rule wil output one reason code per row based on what signal makes the most contribution to the score, from the following:
1 - Stale, good position but low click through rate
2 - Only stale
3 - Low click through rate gap only

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

fact = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet",
    storage_options={"token": os.environ["HF_TOKEN"]}
)
content_dates = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet",
    storage_options={"token": os.environ["HF_TOKEN"]},
    columns=["client_hash_id", "content_hash_id", "content_updated_date", "word_count"]
)
fact = fact.merge(content_dates, on=["client_hash_id", "content_hash_id"], how="left")

avail = fact[fact["gsc_data_available"] == True].copy()
avail["report_date"] = pd.to_datetime(avail["report_date"])
avail["content_updated_date"] = pd.to_datetime(avail["content_updated_date"], errors="coerce")

avail["days_since_update"] = np.where(
    avail["content_updated_date"] <= avail["report_date"],
    (avail["report_date"] - avail["content_updated_date"]).dt.days,
    np.nan
)

avail["ctr"] = np.where(avail["gsc_impressions"] > 0, avail["gsc_clicks"] / avail["gsc_impressions"], np.nan)

agg = avail.groupby(["client_hash_id", "content_hash_id"]).agg(
    total_clicks=("gsc_clicks", "sum"),
    total_impressions=("gsc_impressions", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_since_update=("days_since_update", "max"),
    word_count=("word_count", "first"),
).reset_index()

agg["ctr"] = np.where(agg["total_impressions"] > 0, agg["total_clicks"] / agg["total_impressions"], 0)

staleness_score = (agg["days_since_update"].fillna(0) / 365).clip(0, 1)
position_score  = (1 - (agg["avg_position"].fillna(100) / 100).clip(0, 1))

MIN_IMPRESSIONS = 20
has_enough_traffic = agg["total_impressions"] >= MIN_IMPRESSIONS
ctr_gap_score = (1 - agg["ctr"].fillna(0)).where(has_enough_traffic, 0)

agg["score"] = staleness_score * 40 + position_score * 30 + ctr_gap_score * 30

def assign_reason(row):
    stale = row["days_since_update"] >= 90 if pd.notna(row["days_since_update"]) else False
    good_position = row["avg_position"] <= 20
    enough_traffic = row["total_impressions"] >= MIN_IMPRESSIONS
    low_ctr = row["ctr"] < 0.02 and enough_traffic
    if stale and good_position and low_ctr:
        return "STALE_LOW_CTR_GOOD_POSITION"
    elif stale:
        return "STALE_ONLY"
    elif good_position and low_ctr:
        return "CTR_GAP_ONLY"
    else:
        return "OTHER"

agg["reason_code"] = agg.apply(assign_reason, axis=1)
agg["action"] = "refresh_title_meta"

queue = agg.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("queue rows:", len(queue))
print("rows with unknown staleness (NaN days_since_update):", agg["days_since_update"].isna().sum())
print(queue.head(20)[["content_hash_id","score","avg_position","ctr","total_impressions","days_since_update","reason_code","action"]])

queue rows: 176738
rows with unknown staleness (NaN days_since_update): 148852
                 content_hash_id      score  avg_position       ctr  \
145840  content_3a6cca86385b5e98  88.743630      3.183333  0.000000   
145880  content_42ce26be1ec6be00  87.611934      4.262553  0.001360   
102816  content_ee812ce364842971  87.317870      5.378788  0.000000   
102621  content_d48bc24ef6eacabb  87.110717      6.069298  0.000000   
104764  content_11789763d6d348a6  85.430348      1.076923  0.000000   
145939  content_5120dcbbb086843d  85.172141      6.321173  0.000000   
104919  content_12ec6a93b74b6246  84.884675      2.895833  0.000000   
146411  content_bc5c5010efa1afe0  84.293642      4.865942  0.000000   
105435  content_175ece404bc4d2ba  83.993836      5.500000  0.000000   
110545  content_468cf8de0311e42f  83.938870      7.875000  0.000000   
146287  content_9dc017d4ef83c0d9  83.885366      6.226862  0.000000   
102924  content_fb428c6e1ca78da4  83.777856      4.103634  0.010204  

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
Format: action / reason code / confidence note / what would make it wrong

Top-20 review:

1. content_3a6cca86 — refresh title data / stale, good rank, no clicks /
   low confidence, only 25 views / wrong if page just doesn't get clicked
   no matter what

2. content_42ce26be — refresh title data / stale, good rank, no clicks /
   high confidence, 4,411 views / wrong if the page content doesn't match
   what people are searching for

3. content_ee812ce3 — refresh title data / stale, good rank, no clicks /
   low confidence, only 21 views / wrong if it's old content still ranking
   by accident, not worth keeping

4. content_d48bc24e — refresh title data / stale, good rank, no clicks /
   medium confidence, 48 views / wrong if it's a branded search where people
   already know where they're going

5. content_11789763 — refresh title data / stale, top rank, no clicks /
   medium confidence, 22 views / wrong if Google shows the answer directly,
   so nobody needs to click

6. content_5120dcbb — refresh title data / stale, good rank, no clicks /
   high confidence, 1,429 views / wrong if the page topic doesn't match
   what the searcher actually wants

7. content_12ec6a93 — refresh title data / stale, good rank, no clicks /
   low confidence, only 27 views / wrong if the client meant to
   deprioritize this page on purpose

8. content_bc5c5010 — refresh title data / stale, good rank, no clicks /
   medium confidence, 69 views / wrong if a competitor is grabbing the
   clicks in the search results

9. content_175ece40 — refresh title data / stale, good rank, no clicks /
   low confidence, only 20 views (right at our minimum) / wrong if it's
   just a low-traffic keyword, not a real problem

10. content_468cf8de — refresh title data / stale, good rank, no clicks /
    medium confidence, 45 views / wrong if the ranking spot is low enough
    that people don't scroll to see it

11. content_9dc017d4 — refresh title data / stale, good rank, no clicks /
    high confidence, 231 views / wrong if other stuff on the search page
    (like images or FAQs) is taking the clicks

12. content_fb428c6e — refresh title data / stale, good rank, low clicks /
    high confidence, 490 views / wrong if 1% CTR is actually normal for
    this kind of search

13. content_b86e00f8 — refresh title data / stale, good rank, low clicks /
    medium-high confidence, 170 views / wrong if the title was already
    updated recently and this is the new normal

14. content_eba53d72 — refresh title data / stale, good rank, low clicks /
    high confidence, 734 views / wrong if the page doesn't actually match
    what people are searching for

15. content_d91b572e — refresh title data / stale, good rank, no clicks /
    medium-high confidence, 130 views / wrong if the whole page is outdated,
    not just the title

16. content_6ab29e52 — refresh title data / stale, good rank, low clicks /
    high confidence, 486 views / wrong if the client has another page
    competing for the same keyword

17. content_bea86ce3 — refresh title data / stale, good rank, very low
    clicks / high confidence, 3,670 views / wrong if the ranking keyword
    barely relates to the page's real topic

18. content_2879353f — refresh title data / stale, good rank, no clicks /
    medium confidence, 86 views / wrong if the title was actually updated
    recently but our data didn't catch it

19. content_e3faf067 — refresh title data / stale, good rank, no clicks /
    medium-high confidence, 218 views / wrong if this page's ranking spot
    jumps around a lot month to month

20. content_a9e5a8f1 — refresh title data / stale, good rank, low clicks /
    high confidence, 482 views / wrong if the client has a near-duplicate
    page splitting the clicks


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(queue.head(20)[["content_hash_id","score","avg_position","ctr","total_impressions","days_since_update","reason_code","action"]])

                 content_hash_id      score  avg_position       ctr  \
145840  content_3a6cca86385b5e98  88.743630      3.183333  0.000000   
145880  content_42ce26be1ec6be00  87.611934      4.262553  0.001360   
102816  content_ee812ce364842971  87.317870      5.378788  0.000000   
102621  content_d48bc24ef6eacabb  87.110717      6.069298  0.000000   
104764  content_11789763d6d348a6  85.430348      1.076923  0.000000   
145939  content_5120dcbbb086843d  85.172141      6.321173  0.000000   
104919  content_12ec6a93b74b6246  84.884675      2.895833  0.000000   
146411  content_bc5c5010efa1afe0  84.293642      4.865942  0.000000   
105435  content_175ece404bc4d2ba  83.993836      5.500000  0.000000   
110545  content_468cf8de0311e42f  83.938870      7.875000  0.000000   
146287  content_9dc017d4ef83c0d9  83.885366      6.226862  0.000000   
102924  content_fb428c6e1ca78da4  83.777856      4.103634  0.010204   
146400  content_b86e00f8dae4dd5c  83.764341      5.676747  0.005882   
102798

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Initially for the rule it was favoring pages with only a few views as 61% of all pages had 0% CTR but a quarter of these only had 6 views or less. So basically it was treating no data as bad CTR which was not correct.
So to fix it I added a minimum of 20 impressions before considering the CTR number. After this fix the top 20 pages have real traffic between 20 and 4411 instead of very little views however 20 is still a bit low.

Leakage check: I found out that content_updated_date was coming after report_date in more than 80% of rows so I fixed this to prevent leakage.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
future_updates = avail[avail["content_updated_date"] > avail["report_date"]]
print("rows where content_updated_date is AFTER report_date:", len(future_updates))

print("of those, how many now have NaN days_since_update (fix working):",
      future_updates["days_since_update"].isna().sum())

rows where content_updated_date is AFTER report_date: 2962316
of those, how many now have NaN days_since_update (fix working): 2962316


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.